<b><font size="5" color="#E8800A">Notebook 1 · Data Exploration </font></b><br>
<b><font size="6">Importação e análise inicial dos dados em bruto</font></b><br>

**Objetivo.** Observar e diagnosticar os dados em bruto (`train.csv`, `test.csv` e `clientes.csv`): estrutura, qualidade, distribuições, relação com o target `repeat_purchase_90d`, estrutura temporal e ligação ao `clientes.csv`.

**O que este notebook não faz.** Não altera, limpa nem transforma os dados. A limpeza, a integração do `clientes.csv`, o encoding e a engenharia de features ficam para o `notebook_02_preprocessing`.

**Output.** Uma lista de problemas detetados e das decisões propostas para o notebook 02 (secção 5), cada uma com a evidência que a sustenta.

**Regra anti-fuga de informação.** O `test.csv` é usado apenas para comparar distribuições com o treino. Nada dele é usado para ajustar transformações, e `train` e `test` são mantidos em variáveis separadas ao longo de todo o notebook.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>

* [<font color='#E8800A'>1 - Setup e Carregamento</font>](#sc)
* [<font color='#E8800A'>2 - Visão geral e dicionário de dados</font>](#visao)
* [<font color='#E8800A'>3 - Qualidade dos dados</font>](#qualidade)
* [<font color='#E8800A'>4 - Target e desequilíbrio de classes</font>](#target)
* [<font color='#E8800A'>5 - Estatísticas descritivas e distribuições</font>](#estatistica)
* [<font color='#E8800A'>6 - Relações com o target</font>](#relacao)
* [<font color='#E8800A'>7 - Estrutura temporal, repetição de clientes e deriva train vs. test</font>](#temporaç)
* [<font color='#E8800A'>8 - Análise crítica de identificadores e ruído</font>](#analise)
* [<font color='#E8800A'>9 - Ligação com o clientes.csv</font>](#clientes)
* [<font color='#E8800A'>5 - Conclusões e lista de decisões para o notebook 02</font>](#conclusao)

</div>

# <font color='#E8800A'>1 - Setup e Carregamento</font><a class="anchor" id="sc"></a>

[Back to TOC](#toc)

Importação das bibliotecas necessárias para o desenvolvimento e análise dos dados neste NB, nomeadamente o pandas

In [75]:
import pandas as pd

Leitura dos ficheiros clientes.csv, train.csv, test.csv e sample_submission.csv, que serão utilizados ao longo do projeto.

In [76]:
clientes_data = pd.read_csv("../data/Raw/clientes.csv")

sample_sub = pd.read_csv("../data/Raw/sample_submission.csv")

test_data = pd.read_csv("../data/Raw/test.csv")

train_data = pd.read_csv("../data/Raw/train.csv")

# <font color='#E8800A'>2 - Visão geral e dicionário de dados</font><a class="anchor" id="visao"></a>

[Back to TOC](#toc)

<b><font size="6">clientes.csv
</font></b><br>

É o ficheiro que contém informação cadastral dos clientes. Não representa diretamente uma compra.

In [77]:
print("Dimensões do dataset:")
print(clientes_data.shape)

print("\nPrimeiras 5 linhas:")
display(clientes_data.head(5))

print("\nTipos de dados:")
display(clientes_data.dtypes)

Dimensões do dataset:
(1143, 13)

Primeiras 5 linhas:


,customer_id,country_or_region,commercial_region,island,municipality,city,postal_area,postal_code,street,address,customer_type,company_name,customer_since_year
0,842570,Country CT11: Land of The Final Version,Region RG37: We Will See Later,NaN,NaN,City CY417: Missing Cell Port,NaN,Postal code PC828: #REF! Delivery Route,NaN,Address AD1120: Cell Without a Home,NaN,Excel Is Not A Database Ltd 150,2018



Tipos de dados:


customer_id    int64
               ...  
Length: 13, dtype: object

Contém informações acerca da:  Identificação do cliente;
Localização (country_or_region, city, municipality, etc.);
Endereço;
Tipo de cliente;
Nome da empresa;
Ano em que o cliente começou a relação (customer_since_year).

<b><font size="6">sample_sub.csv
</font></b><br>

É um ficheiro de exemplo do formato que a submissão final deve ter.

In [78]:
print("Dimensões do dataset:")
print(sample_sub.shape)

print("\nPrimeiras 5 linhas:")
display(sample_sub.head(5))

print("\nTipos de dados:")
display(sample_sub.dtypes)

Dimensões do dataset:
(986, 2)

Primeiras 5 linhas:


,ID,repeat_purchase_90d
0,10000006,1



Tipos de dados:


ID    int64
      ...  
Length: 2, dtype: object

Contém apenas o ID, e repeat_purchase_90d

<b><font size="6">test_data.csv
</font></b><br>

É utilizado posteriormente para gerar as previsões do modelo.

In [79]:
print("Dimensões do dataset:")
print(test_data.shape)

print("\nPrimeiras 5 linhas:")
display(test_data.head(5))

print("\nTipos de dados:")
display(test_data.dtypes)

Dimensões do dataset:
(986, 38)

Primeiras 5 linhas:


,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,days_since_previous_purchase,customer_tenure_days,prior_purchase_count,prior_total_spend,prior_mean_purchase_value,prior_std_purchase_value,prior_min_purchase_value,prior_max_purchase_value,prior_mean_gap_days,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,ID,random_noise
0,286.37,582.05,232.82,53.55,0.0,0.0,349.23,104.409267,22,1,40.0,3905,122,46287.23,379.403525,528.297548,2.72,3198.0,31.942149,48.035282,0,0.0,6,1295.82,9,2498.9,23,9140.12,519184,Maybe Tomorrow Payment 12,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 80,Final Excel Warehouse 1,Ctrl Z Series 2,2026-08-22,10000006,0.621602



Tipos de dados:


Total a Pagar    float64
                  ...   
Length: 38, dtype: object

Contém informação sobre as compras, incluindo: valores da compra;
descontos e custos;
informação temporal;
histórico de compras do cliente;
comportamento nos últimos 30, 90, 180 e 365 dias;
características da transação;
customer_id;
e, principalmente, o target.
Tem essencialmente a mesma estrutura do train.csv, mas não contém repeat_purchase_90d.

<b><font size="6">train_data.csv
</font></b><br>

É o dataset principal de treino.

In [80]:
print("Dimensões do dataset:")
print(train_data.shape)

print("\nPrimeiras 5 linhas:")
display(train_data.head(5))

print("\nTipos de dados:")
display(train_data.dtypes)

Dimensões do dataset:
(6534, 39)

Primeiras 5 linhas:


,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,repeat_purchase_90d,days_since_previous_purchase,customer_tenure_days,prior_purchase_count,prior_total_spend,prior_mean_purchase_value,prior_std_purchase_value,prior_min_purchase_value,prior_max_purchase_value,prior_mean_gap_days,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,ID,random_noise
0,558.11,510.9,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,1,79.0,126.0,2.0,858.3,429.15,112.656252,349.49,508.81,47.0,NaN,0.0,0.0,1.0,349.49,2.0,858.3,2.0,858.3,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,10000000,-0.660448



Tipos de dados:


Total a Pagar    float64
                  ...   
Length: 39, dtype: object

Contém informação sobre as compras, incluindo: valores da compra;
descontos e custos;
informação temporal;
histórico de compras do cliente;
comportamento nos últimos 30, 90, 180 e 365 dias;
características da transação;
customer_id;
e, principalmente, o target.

<b><font size="6">Dicionario de dados:
</font></b><br>
| Grupo       | Colunas |
|-------------|---------|
| Monetárias  | `Total a Pagar`, `Total Bruto`, `Total Liquido`, `Total Imp_`, `Total Portes`, `Total Desc_ Global`, `Total Desc_ Linha`, `Custo`, `prior_total_spend`, `prior_mean_purchase_value`, `prior_std_purchase_value`, `prior_min_purchase_value`, `prior_max_purchase_value`, `spend_30d`, `spend_90d`, `spend_180d`, `spend_365d` |
| Detalhe     | `N_ Detalhes`, `invoice_count_current_purchase` |
| Categóricas | `payment_type`, `salesperson`, `commercial_zone`, `transport_method`, `warehouse`, `document_series` |
| Timing      | `purchase_date`, `days_since_previous_purchase`, `customer_tenure_days`, `prior_mean_gap_days`, `prior_std_gap_days` |
| Histórico   | `prior_purchase_count`, `prior_total_spend`, `prior_mean_purchase_value`, `prior_std_purchase_value`, `prior_min_purchase_value`, `prior_max_purchase_value`, `prior_mean_gap_days`, `prior_std_gap_days` |
| Rolling     | `purchase_count_30d`, `spend_30d`, `purchase_count_90d`, `spend_90d`, `purchase_count_180d`, `spend_180d`, `purchase_count_365d`, `spend_365d` |
| Noise       | `random_noise` |
| Identificador | `customer_id`, `ID` |
| Target      | `repeat_purchase_90d` |

# <font color='#E8800A'>3 - Qualidade dos Dados</font><a class="anchor" id="qualidade"></a>

[Back to TOC](#toc)

depois de olhar para os dados que temos verificaremos valores em falta, duplicados, ids duplicados, valores implausiveis, consistencias entre colunas relacionadas. no fim o registo.

<b><font size="6">Valores em falta:</font></b><br>


In [66]:
print("valores em falta em clientes_data:")
display(clientes_data.isna().sum())

print("valores em falta em sample_sub:")
display(sample_sub.isna().sum())

print("valores em falta em test_data:")
display(test_data.isna().sum())

print("valores em falta em train_data:")
display(train_data.isna().sum())



valores em falta em clientes_data:


customer_id    0
              ..
Length: 13, dtype: int64

valores em falta em sample_sub:


ID    0
     ..
Length: 2, dtype: int64

valores em falta em test_data:


Total a Pagar    0
                ..
Length: 38, dtype: int64

valores em falta em train_data:


Total a Pagar    65
                 ..
Length: 39, dtype: int64

com isto concluimos que apenas faltam valores em clientes_data e train_data

<b><font size="6">Valores Duplicados:</font></b><br>


In [67]:
duplicados = pd.DataFrame({
    "Dataset": ["clientes", "sample_submission", "test", "train"],
    "Linhas duplicadas": [
        clientes_data.duplicated().sum(),
        sample_sub.duplicated().sum(),
        test_data.duplicated().sum(),
        train_data.duplicated().sum()
    ]
})

display(duplicados)

,Dataset,Linhas duplicadas
0,clientes,0


nao se verificam valores duplicados no dataset

<b><font size="6">Valores Implausíveis:</font></b><br>


Avaliação de existencia de valores negativos porque neste contexo nao pode haver (trata-se de dinheiro). apenas serao feitos no train devido ao processamento ser unicamente feito nesta bd, assumindo que se gouver falhas no de teste, será para teste proposital do modelo

In [68]:
# Colunas onde não são esperados valores negativos
colunas_numericas = [
    "Total a Pagar",
    "Total Bruto",
    "Total Liquido",
    "Total Imp_",
    "Total Portes",
    "Total Desc_ Global",
    "Total Desc_ Linha",
    "Custo",
    "N_ Detalhes",
    "invoice_count_current_purchase",
    "days_since_previous_purchase",
    "customer_tenure_days",
    "prior_purchase_count",
    "prior_total_spend",
    "prior_mean_purchase_value",
    "prior_std_purchase_value",
    "prior_min_purchase_value",
    "prior_max_purchase_value",
    "prior_mean_gap_days",
    "prior_std_gap_days",
    "purchase_count_30d",
    "spend_30d",
    "purchase_count_90d",
    "spend_90d",
    "purchase_count_180d",
    "spend_180d",
    "purchase_count_365d",
    "spend_365d"
]

print("Valores negativos no TRAIN:")
negativos_train = (train_data[colunas_numericas] < 0).sum()
negativos_train = negativos_train[negativos_train > 1]

if len(negativos_train) > 0:
    display(negativos_train)
else:
    print("Valores conforme: não existem colunas com mais de 1 valor negativo.")


Valores negativos no TRAIN:


days_since_previous_purchase    16
                                ..
Length: 4, dtype: int64

comparação dos gastos aos 30 90 180 e 365 dias

In [73]:
print("TRAIN - Inconsistências nas contagens:")
display(
    train_data[
        (train_data["purchase_count_30d"] > train_data["purchase_count_90d"]) |
        (train_data["purchase_count_90d"] > train_data["purchase_count_180d"]) |
        (train_data["purchase_count_180d"] > train_data["purchase_count_365d"])
    ]
)

TRAIN - Inconsistências nas contagens:


,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,repeat_purchase_90d,days_since_previous_purchase,customer_tenure_days,prior_purchase_count,prior_total_spend,prior_mean_purchase_value,prior_std_purchase_value,prior_min_purchase_value,prior_max_purchase_value,prior_mean_gap_days,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,ID,random_noise


Havia mais verificações a fazer mas seriam imensas e estariamos a usar essas verificações de forma inadequada para tomar decisões demasiado específicas.